In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:

# ------------------------------------------------------------
# 1. Load simulations and parameter sets
# ------------------------------------------------------------
project_dir = "/path/to/directory"
sim_path = f"{project_dir}/data/apple_simulations.csv"
param_path = f"{project_dir}/data/apple_DELSA_LHS_with_local_sensitivity_runs.csv"
fig_dir = f"{project_dir}/Figures"

sim = pd.read_csv(sim_path)
param = pd.read_csv(param_path)

In [4]:

# The analysis target is absolute FST bias.
sim["Abs_Bias"] = sim["Bias"].abs()
sim = sim.sort_values("run_id", kind="stable")

# The observation ordering must be identical within every simulation run.
sim_obs = sim.loc[sim["run_id"] == 1, ["Station", "bin15"]].reset_index(drop=True)
sim_obs["bin15"] = pd.to_datetime(sim_obs["bin15"], format="mixed")
sim_obs["Hour"] = sim_obs["bin15"].dt.hour

stations = ["Sunrise", "Quincy", "Snipes_Rd."]

In [5]:
# Peak: 12:00–18:00 (inclusive); remaining: all other times
peak_mask = (
    (sim_obs["bin15"].dt.hour >= 12) &
    (
        (sim_obs["bin15"].dt.hour < 18) |
        (
            (sim_obs["bin15"].dt.hour == 18) &
            (sim_obs["bin15"].dt.minute == 0)
        )
    )
).to_numpy()

remaining_mask = ~peak_mask

run_ids = sim["run_id"].drop_duplicates().to_numpy()

abs_bias_matrix = sim["Abs_Bias"].to_numpy(dtype=np.float32).reshape(
    len(run_ids), len(sim_obs)
)

run_pos = {run_id: i for i, run_id in enumerate(run_ids)}

In [6]:

# ------------------------------------------------------------
# 2. Parameter uncertainty and base/perturbed-run pairing
# ------------------------------------------------------------
parameter_bounds = {
    "fruit_albedo":             (0.30, 0.70),
    "fruit_emissivity":         (0.93, 0.98),
    "fruit_diameter_m":         (0.06, 0.10),
    "canopy_resistance_s_m":    (6500.0, 20000.0),
    "diffuse_sw_sky_view":      (0.80, 1.00),
    "longwave_sky_view":        (0.80, 1.00),
    "ground_reflected_sw_view": (0.00, 0.30),
    "air_temp_bias_C":          (-4.0, 1.0),
    "rh_bias_pct":              (-10.0, 30.0),
    "wind_speed_multiplier":    (0.40, 1.10),
}

param_cols = list(parameter_bounds)

# Variance of each independently uniform parameter.
param_variance = pd.Series({
    p: (hi - lo)**2 / 12
    for p, (lo, hi) in parameter_bounds.items()
})

base = param.loc[
    param["run_type"] == "base", ["run_id", "base_sample_id"] + param_cols
].copy()
base = base.rename(columns={
    "run_id": "base_run_id",
    **{p: f"{p}_base" for p in param_cols}
})

local = param.loc[
    param["run_type"] == "local_sensitivity",
    ["run_id", "base_sample_id", "perturbed_parameter"] + param_cols
].copy()
local = local.rename(columns={"run_id": "perturbed_run_id"})

pairs = local.merge(base, on="base_sample_id", how="left")
pairs["x_base"] = [
    row[f"{row['perturbed_parameter']}_base"] for _, row in pairs.iterrows()
]
pairs["x_pert"] = [
    row[row["perturbed_parameter"]] for _, row in pairs.iterrows()
]
pairs["dx"] = pairs["x_pert"] - pairs["x_base"]

# A zero perturbation would make the finite-difference derivative undefined.
if (pairs["dx"] == 0).any():
    raise ValueError("Found zero-sized parameter perturbations (dx = 0).")



In [7]:

# ------------------------------------------------------------
# 3. Observation-level DELSA, pooling all three stations
# ------------------------------------------------------------
def run_combined_delsa(time_mask):
    area_results = []
    cdf_results = []
    q_grid = np.linspace(0, 1, 101)

    # Pool observations only after calculating each observation's sensitivity.
    obs_mask = time_mask & sim_obs["Station"].isin(stations).to_numpy()

    for sid in range(1, 501):
        pset = pairs[pairs["base_sample_id"] == sid]
        base_run = int(pset["base_run_id"].iloc[0])
        y_base = abs_bias_matrix[run_pos[base_run], obs_mask]

        local_variance = []
        names = []

        for _, row in pset.iterrows():
            p = row["perturbed_parameter"]
            pert_run = int(row["perturbed_run_id"])
            y_pert = abs_bias_matrix[run_pos[pert_run], obs_mask]

            derivative = (y_pert - y_base) / row["dx"]
            v = derivative**2 * param_variance[p]

            local_variance.append(v)
            names.append(p)

        local_variance = np.vstack(local_variance)
        total = local_variance.sum(axis=0)
        S = np.divide(
            local_variance,
            total,
            out=np.full_like(local_variance, np.nan),
            where=total > 0
        )

        for j, p in enumerate(names):
            sensitivity = S[j]
            sensitivity = sensitivity[np.isfinite(sensitivity)]

            # One area value and one CDF per base set and parameter.
            area_results.append({
                "base_sample_id": sid,
                "parameter": p,
                "area": sensitivity.mean()
            })

            x = np.quantile(sensitivity, q_grid)
            for xx, yy in zip(x, q_grid):
                cdf_results.append({
                    "base_sample_id": sid,
                    "parameter": p,
                    "sensitivity": xx,
                    "cdf": yy
                })

    return pd.DataFrame(area_results), pd.DataFrame(cdf_results)


area_df_peak, cdf_df_peak = run_combined_delsa(peak_mask)
area_df_remaining, cdf_df_remaining = run_combined_delsa(remaining_mask)

In [8]:

# Percentage-scale summary for each period.
for period, area_df in [
    ("Peak (12:00–18:59)", area_df_peak),
    ("Remaining (19:00–11:59)", area_df_remaining),
]:
    summary = (
        area_df.groupby("parameter")["area"]
        .agg(["mean", "median", "std"])
        .sort_values("mean", ascending=False) * 100
    )
    print(f"\n{period}: area above CDF (%)")
    print(summary.round(2))

In [9]:

# ------------------------------------------------------------
# 4. Plot boxplots and all 500 individual CDFs per parameter
# ------------------------------------------------------------
plt.rcParams.update({
    "font.size": 13,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "axes.linewidth": 0.9,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

labels = {
    "fruit_albedo": r"$\alpha_f$",
    "air_temp_bias_C": r"$\Delta T_{air}$",
    "wind_speed_multiplier": r"$M_u$",
    "longwave_sky_view": r"$F_a$",
    "fruit_diameter_m": r"$D_f$",
    "diffuse_sw_sky_view": r"$F_d$",
    "rh_bias_pct": r"$\Delta RH$",
    "ground_reflected_sw_view": r"$F_r$",
    "canopy_resistance_s_m": r"$r_c$",
    "fruit_emissivity": r"$\varepsilon_f$",
}

param_order = [
    "fruit_albedo",
    "air_temp_bias_C",
    "wind_speed_multiplier",
    "longwave_sky_view",
    "fruit_diameter_m",
    "diffuse_sw_sky_view",
    "rh_bias_pct",
    "ground_reflected_sw_view",
    "canopy_resistance_s_m",
    "fruit_emissivity",
]

top4 = [
    "fruit_albedo",
    "air_temp_bias_C",
    "wind_speed_multiplier",
    "longwave_sky_view",
]


In [10]:

def plot_station_combined_delsa(area_df, cdf_df, color, output_path):
    fig = plt.figure(figsize=(15.5, 8.6))
    gs = fig.add_gridspec(
        2, 4, height_ratios=[1.00, 1.08], hspace=0.30, wspace=0.12
    )

    # (a) One boxplot per parameter, using 500 areas.
    ax = fig.add_subplot(gs[0, :])
    x = np.arange(len(param_order))
    data = [
        area_df.loc[area_df["parameter"] == p, "area"].values * 100
        for p in param_order
    ]

    bp = ax.boxplot(
        data,
        positions=x,
        widths=0.48,
        patch_artist=True,
        showfliers=True,
        manage_ticks=False,
        whis=1.5
    )

    for box in bp["boxes"]:
        box.set(facecolor=color, edgecolor=color, alpha=0.55, linewidth=1.1)
    for whisker in bp["whiskers"]:
        whisker.set(color=color, linewidth=1.0)
    for cap in bp["caps"]:
        cap.set(color=color, linewidth=1.0)
    for median in bp["medians"]:
        median.set(color="black", linewidth=1.5)
    for flier in bp["fliers"]:
        flier.set(
            marker="o", markersize=3, markerfacecolor="none",
            markeredgecolor=color, markeredgewidth=0.8, alpha=0.70
        )

    ax.set_xticks(x)
    ax.set_xticklabels([labels[p] for p in param_order], fontsize=14)
    ax.set_ylabel(r"Area above CDF")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.30)
    ax.text(
        0.008, 0.97, "(a)", transform=ax.transAxes,
        ha="left", va="top", fontsize=12, fontweight="bold"
    )

    # (b-e) Draw all 500 individual CDFs for each of the top four parameters.
    for i, p in enumerate(top4):
        ax = fig.add_subplot(gs[1, i])
        d = cdf_df[cdf_df["parameter"] == p]

        for sid, group in d.groupby("base_sample_id"):
            group = group.sort_values("cdf")
            ax.plot(
                group["sensitivity"], group["cdf"],
                color=color, linewidth=0.5, alpha=0.08
            )

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xlabel(r"First-order DELSA sensitivity, $S_j^L$")
        if i == 0:
            ax.set_ylabel("Cumulative fraction of observations")
        else:
            ax.tick_params(labelleft=False)

        ax.set_title(f"({chr(98 + i)}) {labels[p]}", loc="left", fontsize=14, pad=6)
        ax.grid(linestyle="--", linewidth=0.45, alpha=0.22)

    plt.tight_layout()
    plt.savefig(output_path, dpi=400, bbox_inches="tight")
    plt.show()



In [13]:
# Match the original figure outputs, with peak and remaining periods separate.
plot_station_combined_delsa(
    area_df_peak,
    cdf_df_peak,
    color="#D55E00",
    output_path=f"{fig_dir}/DELSA_absolute_FST_bias_peak_Apple.png"
)

In [12]:

# Summary statistics for Figure 6a (peak period)

apple_stats = (
    area_df_peak
    .groupby("parameter")["area"]
    .agg(
        mean="mean",
        median="median",
        std="std",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75)
    )
    .mul(100)
    .sort_values("mean", ascending=False)
    .round(2)
)

apple_stats["IQR"] = apple_stats["q75"] - apple_stats["q25"]

display(apple_stats.round(2))
